# Оригинал или перевод: краулинг данных и NER
В данной тетрадке представлены этапы сбора данных и их предобработки (т.е. NER)

## Импорты

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import os
import pandas as pd
import random
from urllib.parse import urljoin


In [ ]:
!pip install natasha

In [ ]:
import os
from pathlib import Path
from natasha import (
    Segmenter,
    MorphVocab,
    NewsNERTagger,
    NewsEmbedding,
    Doc)

from tqdm import tqdm

## Загрузка и выгрузка архивов из коллаба
Поскольку через коллаб нет необходимости переживать из-за отсутствия доступа к некоторым фанфикам и есть уверенность в производительности

In [ ]:
!apt-get install rar

In [ ]:
!rar a "/content/trans-427_1.rar" "/content/trans1"

In [ ]:
!rar a "/content/trans-427_2.rar" "/content/trans2"

In [ ]:
!rar a "/content/origs-427_1.rar" "/content/origs1"

In [ ]:
!rar a "/content/origs-427_2.rar" "/content/origs2"

In [ ]:
!unzip "/content/trans.zip" -d "/content/"

In [ ]:
!unzip "/content/origs.zip" -d "/content/"

In [ ]:
!rar a "/content/origs_NER.rar" "/content/origs_NER"

In [ ]:
!rar a "/content/trans_NER.rar" "/content/trans_NER"

## Краулер

Обкачка сайта, как полагается, происходила в два этапа:
- Сбор подходящих ссылок со страницы поиска
- Скачивание фанфиков с собранных ссылок

На этапе сбора ссылок так же происходила разметка перевод/оригинал

При этом в некоторых фикак было по несколько глав, так что при наличии содержания функция проходила по каждой ссылке и совмещала главы в итоговом файле, а также фиксировала количество глав, чтобы нам было проще оценивать их среднюю длину.

В файлах мы также сохраняли метаданные фиков: ссылки на них, названия, описания и примечания.

### Получение ссылок на фики

Базовая поисковая ссылка, выдающая все работы по фандомам "Роулинг Джоан, Гарри Поттер (Книги) и "Гарри Поттер (Фильмы и сериалы)". На место {} поставляется номер страницы поиска.

In [ ]:
BASEURL = "https://ficbook.group/fanfiction?filter_fandom%5B0%5D=15&filter_fandom%5B1%5D=16&page={}"

In [ ]:
def get_website_content(url):
    """
    Базовая ф-ция для получения html по ссылке
    """
    try:
        response = requests.get(url)
        response.raise_for_status()
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"Ошибка при выполнении запроса к {url}: {e}")
        return None


def get_urls(url, tag, skip=False) -> dict:
    """
    Парсит страницу с поисковой выдачей и возвращает словарь: {ссылка_на_фанфик: 1/0},
    где 1 — у фанфика есть метка (по умолчанию "Перевод", но я решила это не хардкодить),
    и   0 —  нет.
    Если skip=True, то фанфики с меткой не включаются в результат.
    """
    html = get_website_content(url)
    if not html:
        return {}

    soup = BeautifulSoup(html, 'html.parser')
    cards = soup.find_all('div', class_='card-item')
    result = {}

    for card in cards:
        # Находим ссылку на фанфик
        link_tag = card.find('a', class_='card-item__link')
        if not link_tag or not link_tag.get('href'):
            continue
        fic_url = link_tag['href']

        # Находим все метки
        badge_tags = card.find_all('div', class_='card-item__badge')
        badges = [badge.get_text(strip=True) for badge in badge_tags]

        has_tag = tag in badges

        if skip and has_tag:
            continue  # Пропускаем, если метка есть и skip=True

        result[fic_url] = int(has_tag)

    return result

def scrape_pages(start_page, end_page, output_file='urls.csv', base_url_template=BASEURL, tag="Перевод", skip=False):
    """
    Обходит страницы от start_page до end_page, собирает ссылки с помощью get_urls,
    записывает результат в CSV-файл. Принимает на вход аргументы для get_urls.

    """

    with open(output_file, mode='a', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['url', 'has_tag'])  # Заголовок таблицы

        for page in range(start_page, end_page + 1):
            url = base_url_template.format(page)
            print(f"Обработка страницы {page}...")
            urls_dict = get_urls(url, tag, skip=skip)

            for fic_url, has_tag in urls_dict.items():
                writer.writerow([fic_url, has_tag])

    print(f"Сбор завершён. Результаты в {output_file}")


In [ ]:
scrape_pages(1, 463)  # Через коллаб доступно столько страниц поиска

### Скачивание фанфиков

Очевидно, оригинальных фиков значительно больше, чем переведённых (примерно в 10 раз), поэтому сначала мы качаем все доступные переводы, а потом выбираем столько же случайных оригинальных фанфиков и качаем уже их.

In [ ]:
urls = pd.read_csv("urls.csv")
origs = urls[urls['has_tag'] == 0]['url'].tolist()
trans = urls[urls['has_tag'] == 1]['url'].tolist()

few_origs = random.sample(origs, k=len(trans))
len(few_origs)

856

Сохраним выбранные для скачивания ссылки в отдельных файлах

In [ ]:
with open("origs_urls.txt", mode='a', newline="\n", encoding='utf-8') as outfile:
    for line in few_origs:
        print(line, file=outfile)

with open("trans_urls.txt", mode='a', newline="\n", encoding='utf-8') as outfile:
    for line in trans:
        print(line, file=outfile)

Так же мы посчитали, что в среднем оригинальные тексты будут значительно длиннее, потому что существует достаточно большое количество работ-долгостроев на 500+ глав, которые никогда не будут завершены.

Мы решили ограничивать длину качаемых оригинальных текстов максимальным количеством глав у переводов, решив, что долгострои переводить  никто не возьмётся.

Однако один такой "переводимый вот уже 4 года" фанфик попался, он содержит на настоящий момент 364 главы, мы решили, что он может навредить дообучению из-за того, что он в два с лишним раза длиннее своего ближайшего (единственного) соседа и выбросили его. Если нам будет недостаточно материала для построения классификации, мы вернём его и добавим один такой же долгострой.

Ниже функции для скачивания фиков.

В ней

Вспомогательные функции для извлечения названия, описания, примечаний, текста фанфика, а также записи информации об ошибках в отдельный файл при проблеме с доступом по ссылке или если длина фика выше заданной

In [ ]:
def _extract_title(soup):
    title_tag = soup.find('title')
    return title_tag.get_text(strip=True) if title_tag else None

def _extract_description(soup):
    desc_div = soup.find('div', class_='card-item__field-title', string='Описание')
    if desc_div:
        text_div = desc_div.find_next_sibling('div', class_='card-item__field-text')
        if text_div:
            return text_div.get_text(strip=True)
    return None


def _extract_notes(soup):
    notes_div = soup.find('div', class_='card-item__field-title', string='Примечания')
    if notes_div:
        text_div = notes_div.find_next_sibling('div', class_='card-item__field-text')
        if text_div:
            return text_div.get_text(strip=True)
    return None


def _extract_article_body(soup):
    body_div = soup.find('div', class_='article-parts__description', itemprop='articleBody')
    if body_div:
        return body_div.get_text(separator='\n', strip=True)
    return ""


def _log_error(url, error_code):
    """Записывает ошибку в errors.csv"""
    file_exists = os.path.isfile("errors.csv")
    with open("errors.csv", mode='a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["url", "error_code"])
        writer.writerow([url, error_code])

Основная функция

In [ ]:
def crawl_fanfic(url, max_chapters=None, output_dir="fanfics") -> bool:
    """
    Скачивает фанфик с сайта ficbook.group и сохраняет его в текстовый файл.
    """
    # Создаём папку, если не существует
    os.makedirs(output_dir, exist_ok=True)

    # Извлекаем ID фанфика из URL
    try:
        fanfic_id = url.rstrip('/').split('/')[-1]
        if not fanfic_id.isdigit():
            raise ValueError("Invalid fanfic ID")
    except Exception as e:
        print(f"Неверный формат URL: {url}")
        _log_error(url, 0)
        return False

    html = get_website_content(url)
    if not html:
        print(f"Что-то не так с доступом: {url}")
        _log_error(url, 0)
        return False

    soup = BeautifulSoup(html, 'html.parser')

    # Проверяем наличие блока "Содержание"
    parts_section = soup.find('section', id='parts')
    is_multichapter = parts_section is not None

    if is_multichapter:
        part_links = []
        for li in parts_section.find_all('li', class_='part-item'):
            a_tag = li.find('a', class_='part-item__link', href=True)
            if a_tag:
                full_link = urljoin("https://ficbook.group", a_tag['href'])
                part_links.append(full_link)

        num_chapters = len(part_links)

        if max_chapters is not None and num_chapters > max_chapters:
            print(f"{url} слишком длинный")
            _log_error(url, 1)
            return False

        # Рекурсивно получаем текст каждой части
        all_bodies = []
        for part_url in part_links:
            part_html = get_website_content(part_url)
            if not part_html:
                _log_error(part_url, 0)
                continue
            part_soup = BeautifulSoup(part_html, 'html.parser')
            body = _extract_article_body(part_soup)
            all_bodies.append(body)

        full_text = "\n\n".join(all_bodies)
    else:
        # Одночастный фик
        num_chapters = 1
        full_text = _extract_article_body(soup)

    # Извлекаем метаданные
    title = _extract_title(soup)
    description = _extract_description(soup)
    notes = _extract_notes(soup)

    # Формируем содержимое файла
    lines = [
        url,
        title or "",
        f"Описание: {description}" if description else "",
        f"Примечания: {notes}" if notes else "",
        str(num_chapters),
        "=======================================",
        full_text]

    # Сохраняем
    output_path = os.path.join(output_dir, f"{fanfic_id}.txt")
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write("\n".join(lines))
        return True
    except Exception as e:
        print(f"Ошибка записи файла {output_path}: {e}")
        _log_error(url, 0)
        return False


Непосредственно скачивание фиков (поэтапно, на случай падения кода. Я несильно доверяю коллабу, когда речь идёт о долгой работе)

На наше счастье сайт https://ficbook.group пока что вообще никак себя не защищает от ботов и крулеров, поэтому не возникло ни проблем с капчей, ни потребности в подставном агенте или в рандомной паузе между переходами по ссылкам.

По началу нас порадовала кнопка "скачать фанфик", это значило, что можно было бы попробовать использовать Selenium и получать файлы в хорошем формате автоматически, но на этом прекрасном клоне Фикбука эта кнопка просто не работает)))

In [ ]:
with open("trans_urls.txt", "r", encoding="UTF-8") as tr:
    urls = [x.strip() for x in tr.read().split("\n")]
    print(urls)

In [ ]:
for u in range(0, 427):
        crawl_fanfic(urls[u], max_chapters=None, output_dir="trans1")

In [ ]:
for u in range(427, len(urls)):
    crawl_fanfic(urls[u], max_chapters=None, output_dir="trans2")
    print(urls[u])

In [ ]:
with open("origs_urls.txt", "r", encoding="UTF-8") as orig:
    urls = [x.strip() for x in orig.read().split("\n")]
    print(urls)

In [ ]:
for u in range(0, 427):
    crawl_fanfic(urls[u], max_chapters=151, output_dir="origs1")
    print(u, urls[u])

In [ ]:
for u in range(427, len(urls)):
    crawl_fanfic(urls[u], max_chapters=151, output_dir="origs2")
    print(u, urls[u])

## NER

"Цензурирование" именованных сущностей посредством Natasha как признанного решения задачи NER для русского языка

In [ ]:
# Инициализация Natasha
emb = NewsEmbedding()
segmenter = Segmenter()
morph_vocab = MorphVocab()
ner_tagger = NewsNERTagger(emb)

# Наш разделитель в файлах
SEPARATOR = "======================================="

In [ ]:
def anonymize_text(text):
    """Заменяет именованные сущности на <PER>, <ORG>, <LOC>."""
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_ner(ner_tagger)

    # Сортируем спаны в обратном порядке, чтобы корректно заменять
    for span in sorted(doc.spans, key=lambda x: x.start, reverse=True):
        if span.type in {"PER", "ORG", "LOC"}:
            text = text[:span.start] + f"<{span.type}>" + text[span.stop:]
    return text

In [ ]:
def process_folder(input_folder):
    """Обработка папки целиком: для каждого файла метаданные остаются нетронутыми,
    текст после разделителя анонимизируется,
    файлы называются та же, но помещаются в новую папку"""

    input_path = Path(input_folder)
    output_path = Path(str(input_path) + "_NER")
    output_path.mkdir(exist_ok=True)

    print(f"Обработка папки: {input_folder} → {output_path}")

    txt_files = list(input_path.glob("*.txt"))
    for file_path in tqdm(txt_files, desc=f"Файлы в {input_path.name}"):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()

            parts = content.split(SEPARATOR, 1)
            metadata, body = parts
            processed_body = anonymize_text(body)
            result = metadata + SEPARATOR + "\n" + processed_body

            out_file = output_path / file_path.name
            with open(out_file, 'w', encoding='utf-8') as f:
                f.write(result)

        except Exception as e:
            print(f"Ошибка при обработке {file_path}: {e}")

In [ ]:
process_folder("trans")

In [ ]:
process_folder("origs")